# Weeks 11-12: Eval Science
**Elite AI Systems Engineer 2026 — Thoki Buthelezi**

Benchmarks: GSM8K · HellaSwag · TruthfulQA-MC2  
Model: `Qwen/Qwen2.5-0.5B`  
Hardware: Google Colab T4 (free tier)  
Cost basis: ~$0.10/hr (Colab Pro ~$10/month, T4 ~1 CU/hr)


## 0. Mount Drive

In [ ]:
# mount first so results survive a disconnect
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = "/content/drive/MyDrive/eval_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"output dir: {OUTPUT_DIR}")

## 1. Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

## 2. Install Dependencies

In [ ]:
# don't pin versions — let colab keep its numpy to avoid ABI mismatch
!pip uninstall -y torchvision torchaudio 2>/dev/null
!pip install -q lm-eval transformers accelerate datasets

print("done — restart runtime now, then run from cell 3 onwards")

## 3. Config

In [ ]:
import json, time, datetime, os
import torch

# model
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

# max_gen_toks=256 keeps gsm8k from running ~25s per sample
# default is 2048 which causes colab to disconnect before finishing
MODEL_ARGS = (
    f"pretrained={MODEL_NAME},"
    f"dtype=float16,"
    f"device_map=auto,"
)

# tasks
# gsm8k:          5-shot, exact match, generative
# hellaswag:      10-shot, normalized log-likelihood
# truthfulqa_mc2: 0-shot, multiple choice, no judge model needed
TASKS = [
    {"name": "hellaswag",      "few_shot": 10, "description": "commonsense NLI completion"},
    {"name": "truthfulqa_mc2", "few_shot": 0,  "description": "honesty / truthfulness (MC)"},
    {"name": "gsm8k",          "few_shot": 5,  "description": "grade-school math reasoning"},
]

# cost
COST_PER_HOUR_USD = 0.10  # colab pro T4 basis

# re-declare output dir in case session restarted after cell 0
OUTPUT_DIR = "/content/drive/MyDrive/eval_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"model : {MODEL_NAME}")
print(f"tasks : {[t['name'] for t in TASKS]}")
print(f"gpu   : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")
if torch.cuda.is_available():
    print(f"vram  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 4. Run Evals

In [ ]:
import lm_eval
from lm_eval import evaluator

all_results = {}
run_log = []  # list of {task, duration_s, cost_usd, metric, score}

CHECKPOINT_PATH = f"{OUTPUT_DIR}/checkpoint.json"

def save_checkpoint(log):
    with open(CHECKPOINT_PATH, "w") as f:
        json.dump(log, f, indent=2)
    print(f"  checkpoint saved ({len(log)} tasks done)")

for task_cfg in TASKS:
    task_name = task_cfg["name"]
    few_shot  = task_cfg["few_shot"]
    desc      = task_cfg["description"]

    print(f"\n{'='*55}")
    print(f"  task     : {task_name}")
    print(f"  few-shot : {few_shot}")
    print(f"  desc     : {desc}")
    print(f"{'='*55}")

    t0 = time.time()

    results = evaluator.simple_evaluate(
        model="hf",
        model_args=MODEL_ARGS,
        tasks=[task_name],
        num_fewshot=few_shot,
        batch_size="auto",
        device="cuda",
        log_samples=False,
        verbosity="INFO",
        gen_kwargs="max_new_tokens=256",
        limit=0.25,  
    )

    duration_s  = time.time() - t0
    duration_hr = duration_s / 3600
    cost_usd    = duration_hr * COST_PER_HOUR_USD

    # pull the first numeric non-stderr, non-alias key as the headline metric
    task_res   = results["results"][task_name]
    metric_key = next(
        k for k in task_res
        if not k.endswith("_stderr")
        and k != "alias"
        and isinstance(task_res[k], (int, float))
        )
    metric_score = task_res[metric_key]

    entry = {
        "task":         task_name,
        "description":  desc,
        "few_shot":     few_shot,
        "metric":       metric_key,
        "score":        round(metric_score, 4),
        "duration_s":   round(duration_s, 1),
        "duration_min": round(duration_s / 60, 2),
        "cost_usd":     round(cost_usd, 4),
    }
    run_log.append(entry)
    all_results[task_name] = results

    print(f"\n  {task_name}: {metric_key} = {metric_score:.4f}")
    print(f"  time: {duration_s/60:.1f} min  cost: ${cost_usd:.4f}")

    # save after each task — if colab disconnects mid-run nothing is lost
    save_checkpoint(run_log)

print("\nall tasks done")

## 5. Summarise and Save

In [ ]:
import pandas as pd

df = pd.DataFrame(run_log)

total_cost = df["cost_usd"].sum()
total_min  = df["duration_min"].sum()

print("\n" + "="*60)
print(f"  model  : {MODEL_NAME}")
print(f"  run at : {datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')}")
print("="*60)
print(df[["task", "few_shot", "metric", "score", "duration_min", "cost_usd"]].to_string(index=False))
print("-"*60)
print(f"  total  : {total_min:.1f} min  ${total_cost:.4f} USD")
print("="*60)

timestamp = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M")

csv_path  = f"{OUTPUT_DIR}/results_{timestamp}.csv"
json_path = f"{OUTPUT_DIR}/results_{timestamp}.json"

df.to_csv(csv_path, index=False)

summary = {
    "model":          MODEL_NAME,
    "run_at_utc":     timestamp,
    "cost_basis":     f"${COST_PER_HOUR_USD}/hr (Colab Pro T4)",
    "total_cost_usd": round(total_cost, 4),
    "total_min":      round(total_min, 2),
    "results":        run_log,
}
with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"\n  csv  -> {csv_path}")
print(f"  json -> {json_path}")

## 6. Plot

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f"eval results — {MODEL_NAME}", fontsize=13)

colors = ["#4C72B0", "#55A868", "#C44E52"]
tasks  = df["task"].tolist()
scores = df["score"].tolist()
costs  = df["cost_usd"].tolist()

# scores
bars = axes[0].bar(tasks, scores, color=colors, width=0.5, edgecolor="white", linewidth=1.2)
axes[0].set_ylim(0, 1.0)
axes[0].set_ylabel("score (0-1)")
axes[0].set_title("benchmark scores")
for bar, score in zip(bars, scores):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{score:.3f}",
        ha="center", va="bottom", fontsize=11
    )

# cost breakdown
axes[1].pie(
    costs, labels=tasks, colors=colors,
    autopct=lambda p: f"${p/100*sum(costs):.4f}",
    startangle=140, pctdistance=0.75
)
axes[1].set_title(f"cost breakdown (total: ${sum(costs):.4f})")

plt.tight_layout()
plot_path = f"{OUTPUT_DIR}/eval_plot_{timestamp}.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"plot -> {plot_path}")

## 7. Download

In [ ]:
# files are already in drive 
from google.colab import files
files.download(csv_path)
files.download(json_path)
files.download(plot_path)
print("done — commit to weeks/11-12/ in the monorepo")

---
## Notes

**methodology:**
- truthfulqa_mc2 used instead of generative to avoid needing a judge model
- max_gen_toks=256 to keep gsm8k runtime under 50 min on free T4
- batch_size=auto — harness picks the largest batch that fits in VRAM
- few-shot counts follow the original papers: gsm8k=5, hellaswag=10, truthfulqa=0
- cost basis: colab pro T4 at $0.10/hr

**limitations:**
- qwen2.5-0.5B is a base model, not instruction-tuned — scores reflect pretraining only
- gsm8k exact-match will undercount partially correct chain-of-thought answers
- contamination risk: qwen training data composition is not fully disclosed
